# Atelier 1 — Notebook 2 : Couverture de tagging

**Objectif** : mesurer la qualité du tagging d'un parc cloud à partir du CUR.

Sans tags, **impossible d'attribuer un coût à une équipe** ou un projet. Ce notebook calcule :

1. Le **pourcentage de coûts taggés** par dimension (`team`, `env`, `project`)
2. Les **services orphelins** (coûts sans tag)
3. Une **matrice de couverture** par compte × service

Le résultat sert d'input à la Partie 3 du README (stratégie de répartition des coûts partagés).

In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '${:,.2f}'.format)

## Cellule 1 — Chargement du dataset

On réutilise le même CUR simplifié que le notebook 1. Les colonnes `tag_team`, `tag_env`, `tag_project` peuvent être vides (`NaN` ou chaîne vide) : c'est précisément ce qu'on va mesurer.

In [ ]:
CUR_PATH = Path('../../ressources/datasets/cur_sample.csv')
df = pd.read_csv(CUR_PATH, parse_dates=['usage_date'])

TAG_COLS = ['tag_team', 'tag_env', 'tag_project']
for col in TAG_COLS:
    df[col] = df[col].replace('', pd.NA)

account_labels = {
    '111111111111': 'prod',
    '222222222222': 'staging',
    '333333333333': 'dev',
    '444444444444': 'sandbox',
}
df['account_id'] = df['account_id'].astype(str)
df['account_name'] = df['account_id'].map(account_labels)

print(f'Lignes : {len(df):,}')
print(f'Coût total : ${df.unblended_cost.sum():,.2f}')
df.head()

## Cellule 2 — Taux de couverture global par dimension

Pour chaque tag, on calcule :
- **% lignes taggées** (combien de lignes CUR portent ce tag)
- **% coût taggé** (quelle part du \$ total est couverte)

👉 Le % en coût est **toujours plus important** que le % en lignes : un seul tag manquant sur une grosse instance fait plus mal que dix tags manquants sur des petites lignes.

In [ ]:
total_cost = df['unblended_cost'].sum()
total_rows = len(df)

coverage = []
for col in TAG_COLS:
    tagged = df[df[col].notna()]
    coverage.append({
        'dimension': col.replace('tag_', ''),
        'pct_lignes_taggees': len(tagged) / total_rows * 100,
        'pct_cout_tagge': tagged['unblended_cost'].sum() / total_cost * 100,
        'cout_non_tagge': total_cost - tagged['unblended_cost'].sum(),
    })

coverage_df = pd.DataFrame(coverage)
display(coverage_df)

fig = px.bar(
    coverage_df.melt(id_vars='dimension',
                     value_vars=['pct_lignes_taggees', 'pct_cout_tagge'],
                     var_name='mesure', value_name='pourcentage'),
    x='dimension', y='pourcentage', color='mesure', barmode='group',
    title='Couverture de tagging — % lignes vs % coût',
    labels={'pourcentage': '% couvert', 'dimension': 'Tag'}
)
fig.update_yaxes(range=[0, 100])
fig.show()

## Cellule 3 — Services orphelins

Un **service orphelin** est un service AWS dont une part significative du coût n'est rattachée à aucune équipe (`tag_team` manquant).

On veut connaître, pour chaque service :
- le coût total
- le coût non taggé
- le **% orphelin**

Les services en haut de cette liste sont les premières cibles d'une campagne de tagging.

In [ ]:
by_service = df.groupby('service').agg(
    cout_total=('unblended_cost', 'sum'),
    cout_non_tagge=('unblended_cost',
                    lambda s: s[df.loc[s.index, 'tag_team'].isna()].sum()),
).reset_index()
by_service['pct_orphelin'] = by_service['cout_non_tagge'] / by_service['cout_total'] * 100
by_service = by_service.sort_values('cout_non_tagge', ascending=False)

display(by_service)

fig = px.bar(
    by_service, x='service', y='pct_orphelin',
    hover_data=['cout_total', 'cout_non_tagge'],
    title='% de coût non taggé (team) par service',
    labels={'pct_orphelin': '% orphelin', 'service': 'Service AWS'},
    color='pct_orphelin', color_continuous_scale='Reds'
)
fig.update_yaxes(range=[0, 100])
fig.show()

## Cellule 4 — Matrice de couverture compte × service

Heatmap du **taux de couverture `team`** (en %) croisant comptes et services. Les cases rouges sont les zones à traiter en priorité.

In [ ]:
df['_tagged_team'] = df['tag_team'].notna().astype(int) * df['unblended_cost']

pivot_total = df.pivot_table(
    index='service', columns='account_name',
    values='unblended_cost', aggfunc='sum', fill_value=0
)
pivot_tagged = df.pivot_table(
    index='service', columns='account_name',
    values='_tagged_team', aggfunc='sum', fill_value=0
)
coverage_matrix = (pivot_tagged / pivot_total.replace(0, pd.NA) * 100).fillna(0)

fig = px.imshow(
    coverage_matrix, text_auto='.0f', aspect='auto',
    color_continuous_scale='RdYlGn', zmin=0, zmax=100,
    title='Taux de couverture tag_team par compte × service (%)'
)
fig.show()

df.drop(columns=['_tagged_team'], inplace=True)

## Cellule 5 — Coûts taggés par équipe (showback minimal)

On agrège le coût taggé par équipe : c'est l'embryon d'un **rapport de showback**. Le bloc `non taggé` représente le \$ qu'on ne sait pas attribuer — il faudra le répartir (cf. notebook 03).

In [ ]:
team_costs = (
    df.assign(tag_team=df['tag_team'].fillna('(non taggé)'))
      .groupby('tag_team')['unblended_cost'].sum()
      .reset_index().sort_values('unblended_cost', ascending=False)
)
display(team_costs)

fig = px.pie(
    team_costs, values='unblended_cost', names='tag_team',
    title='Répartition du coût par équipe (incl. non taggé)'
)
fig.show()

## Cellule 6 — Synthèse pour le rapport

Chiffres clés à reporter dans la restitution de l'atelier.

In [ ]:
untagged_team_cost = df[df['tag_team'].isna()]['unblended_cost'].sum()
worst_service = by_service.iloc[0]

print('========== SYNTHÈSE TAGGING ==========')
print(f'Coût total : ${total_cost:,.2f}')
print()
for row in coverage:
    print(f"Couverture {row['dimension']:<8} : "
          f"{row['pct_cout_tagge']:.1f}% du coût ({row['pct_lignes_taggees']:.1f}% des lignes)")
print()
print(f'Coût non rattaché à une équipe : ${untagged_team_cost:,.2f} '
      f'({untagged_team_cost / total_cost * 100:.1f}%)')
print()
print(f"Service le plus orphelin : {worst_service['service']} "
      f"— ${worst_service['cout_non_tagge']:,.2f} non taggés "
      f"({worst_service['pct_orphelin']:.1f}%)")

## 👉 Questions de débrief

1. Quelle dimension de tag est la **moins bien couverte** ? Pourquoi à votre avis ?
2. Y a-t-il des services pour lesquels le tagging est **structurellement impossible** ? (cf. data transfer, services partagés)
3. Si vous deviez fixer un **objectif** de couverture trimestriel, quel chiffre proposeriez-vous et pourquoi ?
4. Comment passeriez-vous d'un constat (ce notebook) à une **action** (campagne de tagging) sans braquer les équipes ?